# Entrega 3 · Modelado predictivo del ROI de una película

**Pregunta del modelo:** con lo que se sabe *antes del estreno* (presupuesto, género, idioma, duración, productora, elenco, fecha), ¿podemos anticipar cuántas veces recupera una película lo que costó?

**Qué cambió desde la Entrega 2**
1. **Dataset:** se volvió a correr el DAG en modo `full`. El notebook lee el CSV más reciente de `include/output/` (`movies_<fecha>.csv`), que genera cada integrante con su propia corrida y no se sube a git. Las salidas guardadas son de la corrida del 24/10/2026: **9.940 películas**, porque 60 no se pudieron bajar por cortes de conexión con TMDb. Si se corre con otro CSV, los números van a variar un poco.
2. **Alcance:** se agregan filtros de montos imposibles (`budget` y `revenue` mayores o iguales a 10.000 USD) y se excluyen las películas de 2026, que siguen en cartelera.
3. **Objetivo:** el ROI, pero se modela y se mide en escala logarítmica: $y = \log_{10}(\text{revenue}/\text{budget}) = \log_{10}(1 + \text{ROI})$. En la sección 3 se muestra por qué el MAE sobre el ROI crudo no sirve para elegir modelos.

El notebook corre de arriba a abajo. La semilla se define en la primera celda y se pasa a todo lo que la acepta.

In [ ]:
# 1. Semilla, librerías y carga del dataset
SEED = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

np.random.seed(SEED)
plt.style.use("default")
plt.rcParams["axes.grid"] = True
pd.set_option("display.float_format", "{:.4f}".format)

from pathlib import Path

# Cada integrante usa el CSV de su propia corrida del DAG (include/output está en .gitignore).
# Se toma el más reciente: el nombre lleva la fecha, así que el mayor alfabético es el último.
CSV_PATH = max(Path("include/output").glob("movies_*.csv"))
df = pd.read_csv(CSV_PATH)
if len(df) < 7_000:
    raise ValueError(f"{CSV_PATH} tiene {len(df):,} filas: parece una corrida en modo subset. "
                     "Correr el DAG con mode=full antes de ejecutar este notebook.")
print(f"scikit-learn {sklearn.__version__} | pandas {pd.__version__}")
print(f"Dataset: {CSV_PATH} -> {df.shape[0]:,} filas x {df.shape[1]} columnas")

## 1. Alcance: qué películas entran

Cuatro filtros. Los de `budget` y año se deciden con columnas de entrada. Los de `revenue` tocan el objetivo, así que se limitan a montos faltantes o imposibles. El de `revenue >= 10.000` se agregó después de una primera corrida (ver sección 9):

| Filtro | Por qué |
|---|---|
| `budget > 0` y `revenue > 0` | Sin los dos montos no hay ROI. No se imputa: inventar un presupuesto es inventar el objetivo. |
| `budget >= 10.000 USD` | Montos de 1, 2 o 7 dólares son errores de carga de TMDb (por ejemplo, *East of Eden* con presupuesto 2). Es un filtro sobre una columna de entrada. |
| `revenue >= 10.000 USD` | Mismo umbral para la recaudación. Lo encontramos al leer los errores del modelo: *Society of the Snow* figura con 1.281 USD y *Memoir of a Snail* con 7 USD. Son estrenos de streaming sin taquilla real o errores de carga, no fracasos. Este filtro toca el objetivo, así que se limita a montos imposibles: saca muy pocas filas y no recorta la cola de éxitos ni de fracasos reales. |
| `release_year < 2026` | Las películas de 2026 siguen en cartelera: su `revenue` está incompleto y su ROI saldría subestimado. |

In [ ]:
# 2. Filtros de alcance e Ingeniería de Features Cinematográficas
pasos = [("Dataset completo", df)]
d = df[(df["budget"] > 0) & (df["revenue"] > 0)]
pasos.append(("Con budget y revenue > 0", d))
d = d[d["budget"] >= 10_000]
pasos.append(("Con budget >= 10.000 USD", d))
d = d[d["revenue"] >= 10_000]
pasos.append(("Con revenue >= 10.000 USD", d))
d = d[d["release_year"] < 2026]
pasos.append(("Estrenadas antes de 2026", d))

for nombre, tabla in pasos:
    print(f"{nombre:<28} {len(tabla):>6,} filas")

data = d.copy()
data["roi"] = (data["revenue"] - data["budget"]) / data["budget"]
data["y"] = np.log10(data["revenue"] / data["budget"])  # = log10(1 + ROI)

# Orden cronológico estricto para evitar fuga en features temporales
data = data.sort_values(["release_year", "release_month", "movie_id"]).reset_index(drop=True)

# --- 2.1 Ratios de Eficiencia de Capital y Sinergias ---
log_b = np.log10(data["budget"])
star_sum = data["lead_actor_popularity"] + data["co_star_popularity"]

data["star_power_ratio"] = star_sum / log_b
data["director_leverage"] = data["director_popularity"] / log_b
data["director_vs_cast"] = (data["director_popularity"] + 1.0) / (data["lead_actor_popularity"] + 1.0)
data["power_pack"] = data["director_popularity"] * data["lead_actor_popularity"]

# --- 2.2 Interacción Género x Escala de Presupuesto ---
median_budget_genre = data.groupby("primary_genre")["budget"].transform("median")
data["budget_vs_genre_median"] = data["budget"] / median_budget_genre
data["is_horror_low_budget"] = ((data["primary_genre"] == "Horror") & (data["budget"] < 5_000_000)).astype(int)

# --- 2.3 Target Encoding Histórico Acumulado por Productora (Sin Data Leakage) ---
# Para cada año t, calcula la mediana del ROI de la productora en años < t
prod_hist_median = []
global_hist_median = []

years = sorted(data["release_year"].unique())
year_to_prod_roi = {}
year_to_global_roi = {}

for y_val in years:
    past = data[data["release_year"] < y_val]
    if len(past) > 0:
        year_to_prod_roi[y_val] = past.groupby("primary_production_company")["y"].median().to_dict()
        year_to_global_roi[y_val] = past["y"].median()
    else:
        year_to_prod_roi[y_val] = {}
        year_to_global_roi[y_val] = data["y"].median()

def get_prod_encoded(row):
    y_val = row["release_year"]
    prod = row["primary_production_company"]
    med_dict = year_to_prod_roi.get(y_val, {})
    if prod in med_dict:
        return med_dict[prod]
    return year_to_global_roi.get(y_val, data["y"].median())

data["prod_hist_roi"] = data.apply(get_prod_encoded, axis=1)

print(f"\nFeatures creadas con éxito. Dimensiones de trabajo: {data.shape}")

## 2. El objetivo: por qué ROI en escala logarítmica

El ROI tiene una cola enorme: la mediana ronda 1,4 (la película recauda 2,4 veces lo que costó), pero hay casos como *Paranormal Activity* con ROI de casi 13.000. En escala lineal, esos pocos casos deciden cualquier promedio. En escala log10, un punto de error significa "le erré por un factor de 10" sin importar si la película es chica o grande.

In [ ]:
# 3. Distribución del ROI crudo vs. log10(revenue/budget)
print(data["roi"].describe(percentiles=[0.25, 0.5, 0.75, 0.99]).to_string())
print("\nLas 5 películas con mayor ROI:")
print(data.nlargest(5, "roi")[["title", "release_year", "budget", "revenue", "roi"]].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(data["roi"], bins=100, color="steelblue")
axes[0].set_yscale("log")
axes[0].set_title("ROI crudo (eje Y en log): una cola de miles")
axes[0].set_xlabel("ROI")
axes[1].hist(data["y"], bins=60, color="forestgreen")
axes[1].axvline(0, color="crimson", linestyle="--", label="Recupera justo lo invertido")
axes[1].set_title("Objetivo del modelo: log10(revenue / budget)")
axes[1].set_xlabel("log10(revenue / budget)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Las cuatro decisiones antes del primer modelo

### 3.1 Métrica: MAE sobre $\log_{10}(\text{revenue}/\text{budget})$

- **Qué error es caro:** para una productora, equivocarse en el *múltiplo* de recuperación. Prever que una película triplica su costo y que la recupere a medias es el mismo error relativo en una de 1 millón que en una de 200 millones.
- **Cómo se lee:** un MAE de 0,40 significa que, en promedio, la predicción del múltiplo `revenue/budget` está errada por un factor de $10^{0,40} \approx 2,5$ (para arriba o para abajo).
- **Por qué no MAE sobre el ROI crudo**, que era la propuesta inicial: lo probamos (celda siguiente). Una sola película (*Paranormal Activity*, ROI ≈ 12.900) cae en un fold y lleva el MAE de ese fold a más de 17, cuando los otros dan 3–7. Con un desvío entre folds tan grande, ningún modelo se distingue del piso: la métrica mide en qué fold cayó el outlier, no qué modelo es mejor.
- **Por qué no RMSE ni R²:** el RMSE eleva al cuadrado y es todavía más sensible a esa cola. El R² no tiene unidades que se puedan explicar a un productor.

### 3.2 Piso: `DummyRegressor(strategy="median")`
Predice siempre la mediana del entrenamiento sin mirar ninguna columna. Es el número contra el que se lee todo lo demás.

### 3.3 Validación que imita el uso
El modelo se usaría para películas **futuras**, con lo aprendido de las pasadas. Por eso:
1. Las películas se ordenan por fecha de estreno (`release_year`, `release_month`).
2. El **20% más reciente** se separa como test y se mira **una sola vez**, al final.
3. Sobre el 80% restante, `TimeSeriesSplit(5)`: cada fold entrena con el pasado y valida con el período siguiente.

Además, las filas **no son independientes**: la misma productora, el mismo director o la misma franquicia se repiten. Un corte al azar mete películas de 2015 en el entrenamiento para predecir una de 2012, y deja secuelas de los dos lados. En la sección 5 se mide cuánto cambia el número si se corta al azar.

### 3.4 Todo adentro de un pipeline
Logaritmos, imputación, escalado y one-hot van dentro de un `ColumnTransformer` que se ajusta en cada fold solo con el entrenamiento de ese fold. Antes de partir solo se calcula el objetivo `y`, que es una cuenta fila por fila y no usa información de otras filas.

In [ ]:
# 4. Prueba descartada: MAE sobre el ROI crudo (para justificar la métrica)
datos_ord = data.sort_values(["release_year", "release_month", "movie_id"]).reset_index(drop=True)
n_test = int(len(datos_ord) * 0.20)
train, test = datos_ord.iloc[:-n_test], datos_ord.iloc[-n_test:]

tss = TimeSeriesSplit(n_splits=5)
dummy_crudo = cross_validate(DummyRegressor(strategy="median"), train, train["roi"], cv=tss,
                             scoring="neg_mean_absolute_error")
mae_crudo = -dummy_crudo["test_score"]
print("Piso con MAE sobre ROI crudo, por fold:", np.round(mae_crudo, 2))
print(f"Media {mae_crudo.mean():.2f} | desvío {mae_crudo.std():.2f}")
for i, (_, idx_val) in enumerate(tss.split(train)):
    fila = train.iloc[idx_val].nlargest(1, "roi").iloc[0]
    print(f"  Fold {i + 1}: mayor ROI = {fila['roi']:>9,.1f} ({fila['title']}, {fila['release_year']})")

## 4. Partición temporal y columnas

In [ ]:
# 5. Qué años cubre cada parte de la validación
print(f"Entrenamiento: {len(train):,} películas ({train['release_year'].min()}-{train['release_year'].max()})")
print(f"Test (20% más reciente, se mira una vez): {len(test):,} películas "
      f"({test['release_year'].min()}-{test['release_year'].max()})\n")
filas = []
for i, (idx_tr, idx_val) in enumerate(tss.split(train)):
    filas.append({
        "fold": i + 1,
        "entrena": f"{train.iloc[idx_tr]['release_year'].min()}-{train.iloc[idx_tr]['release_year'].max()}",
        "n_entrena": len(idx_tr),
        "valida": f"{train.iloc[idx_val]['release_year'].min()}-{train.iloc[idx_val]['release_year'].max()}",
        "n_valida": len(idx_val),
    })
print(pd.DataFrame(filas).to_string(index=False))

| Columna | Rol | Tratamiento dentro del pipeline |
|---|---|---|
| `budget` | Entra | `log10(1 + x)` y escalado. Se conoce antes de filmar. |
| `director_popularity`, `lead_actor_popularity`, `co_star_popularity` | Entran, **bajo sospecha** | `log10(1 + x)` y escalado. Ver sección 8: TMDb da la popularidad *de hoy*, no la previa al rodaje. |
| `runtime`, `release_year` | Entran | 0 se trata como faltante, se imputa con la mediana y se escala. |
| `original_language`, `primary_genre`, `secondary_genre`, `release_month`, `release_day_of_week` | Entran | One-hot; las categorías con menos de 20 películas se agrupan en "otras". |
| `primary_production_company`, `co_production_company` | Entran | One-hot de las productoras con 30 o más películas; el resto, "otras". |
| `revenue` | **Sale** | Es parte del objetivo: fuga directa. |
| `vote_average`, `vote_count` | **Salen** | Existen después del estreno. Se usan solo al final, para auditar las predicciones. |
| `movie_id`, `title` | Salen | Identificadores, no describen la película. |

In [ ]:
# 6. Columnas y preprocesamiento con nuevas features (se reajusta en cada fold)
COLS_LOG = ["budget", "director_popularity", "lead_actor_popularity", "co_star_popularity", "power_pack"]
COLS_NUM = ["runtime", "release_year", "star_power_ratio", "director_leverage", "director_vs_cast", 
            "budget_vs_genre_median", "prod_hist_roi"]
COLS_BOOL = ["is_horror_low_budget"]
COLS_CAT = ["original_language", "primary_genre", "secondary_genre", "release_month", "release_day_of_week"]
COLS_PROD = ["primary_production_company", "co_production_company"]
FEATURES = COLS_LOG + COLS_NUM + COLS_BOOL + COLS_CAT + COLS_PROD


def log10_1p(X):
    return np.log10(1 + np.maximum(np.asarray(X, dtype=float), 0))


def cero_a_nan(X):
    X = np.asarray(X, dtype=float).copy()
    X[X <= 0] = np.nan
    return X


def armar_preprocesador(excluir=()):
    quedan = lambda cols: [c for c in cols if c not in excluir]
    bloques = [
        ("log", make_pipeline(FunctionTransformer(log10_1p, feature_names_out="one-to-one"),
                              StandardScaler()), quedan(COLS_LOG)),
        ("num", make_pipeline(FunctionTransformer(cero_a_nan, feature_names_out="one-to-one"),
                              SimpleImputer(strategy="median"), StandardScaler()), quedan(COLS_NUM)),
        ("bool", "passthrough", quedan(COLS_BOOL)),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20,
                              sparse_output=False), quedan(COLS_CAT)),
        ("prod", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30,
                               sparse_output=False), quedan(COLS_PROD)),
    ]
    return ColumnTransformer([b for b in bloques if b[2]])


def modelo(estimador, excluir=()):
    return Pipeline([("pre", armar_preprocesador(excluir)), ("modelo", estimador)])


X_train, y_train = train[FEATURES], train["y"]
X_test, y_test = test[FEATURES], test["y"]
print(f"{len(FEATURES)} columnas de entrada -> "
      f"{armar_preprocesador().fit_transform(X_train).shape[1]} columnas después del preprocesador")

## 5. Comparación de modelos

Todos con el mismo pipeline y los mismos 5 folds temporales, con sus valores por defecto razonables (el ajuste fino viene en la sección 6):

| Nivel | Modelo | Familia | Qué aporta |
|---|---|---|---|
| 0 | `DummyRegressor(median)` | Piso | Error a batir sin mirar columnas. |
| 1 | `Ridge` | Lineal | Modelo simple de referencia. Con L2 porque el one-hot genera columnas correlacionadas. |
| 2 | `RandomForestRegressor` | Árboles en paralelo (bagging) | No linealidades e interacciones (por ejemplo, género × presupuesto). |
| 3 | `HistGradientBoostingRegressor` | Boosting | Árboles en serie que corrigen el error del anterior. |

In [ ]:
# 7. Validación cruzada temporal con MAE, RMSE y R²
# Incorpora HistGradientBoosting con pérdida Huber (penalización para evitar colapso central)
candidatos = {
    "Piso (mediana)": DummyRegressor(strategy="median"),
    "Ridge": modelo(Ridge(alpha=10.0)),
    "Random Forest": modelo(RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                  max_features=0.3, n_jobs=-1, random_state=SEED)),
    "HistGB (Huber Loss)": modelo(HistGradientBoostingRegressor(loss="huber", quantile=0.85,
                                                               learning_rate=0.04, max_iter=300,
                                                               min_samples_leaf=15, l2_regularization=2.0,
                                                               random_state=SEED)),
}

metricas = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

def evaluar_multimetrica(modelos, cv):
    resultados = []
    for nombre, m in modelos.items():
        r = cross_validate(m, X_train, y_train, cv=cv, scoring=metricas)
        mae_m = -r["test_MAE"].mean()
        rmse_m = -r["test_RMSE"].mean()
        r2_m = r["test_R2"].mean()
        r2_std = r["test_R2"].std(ddof=0)
        resultados.append({
            "Modelo": nombre,
            "MAE medio": mae_m,
            "factor de error": 10 ** mae_m,
            "RMSE medio": rmse_m,
            "R² medio": r2_m,
            "R² desvío": r2_std
        })
    return pd.DataFrame(resultados).set_index("Modelo")

resumen_v2 = evaluar_multimetrica(candidatos, tss)
resumen_v2["mejora MAE vs piso"] = resumen_v2.loc["Piso (mediana)", "MAE medio"] - resumen_v2["MAE medio"]

# Evaluación fold por fold de MAE para comparación pareada
def evaluar_mae_folds(modelos, cv):
    por_fold = {}
    for nombre, m in modelos.items():
        r = cross_validate(m, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
        por_fold[nombre] = -r["test_score"]
    return pd.DataFrame(por_fold, index=[f"fold {i + 1}" for i in range(cv.get_n_splits())])

folds_tss = evaluar_mae_folds(candidatos, tss)

print("=== Resumen Multimétrica (TimeSeriesSplit) ===")
print(resumen_v2.round(4).to_string())
print("\n=== MAE por fold ===")
print(folds_tss.round(4).to_string())

### ¿El modelo complejo le gana al simple por más que el ruido?

El desvío de la tabla anterior mezcla dos cosas: lo que varía el modelo y lo **difícil que es cada período** (el fold 1, que valida con 1988–1997, es más difícil para todos que el fold 3, que valida con 2003–2008). Como todos los modelos se evalúan en los mismos folds, la comparación justa es **fold contra fold**: se resta el MAE de dos modelos en cada fold y se mira la media y el desvío de esa diferencia. Si la mejora media es mayor que su desvío, la mejora no es ruido.

In [ ]:
# 8. Comparación pareada: diferencia de MAE fold por fold
def comparar(a, b):
    dif = folds_tss[a] - folds_tss[b]  # positivo = b se equivoca menos que a
    return {"comparación": f"{b} vs {a}", "mejora media": dif.mean(), "desvío de la mejora": dif.std(ddof=0),
            "folds que gana": f"{(dif > 0).sum()} de {len(dif)}", "supera el ruido": dif.mean() > dif.std(ddof=0)}

pares = [
    ("Piso (mediana)", "Ridge"),
    ("Piso (mediana)", "Random Forest"),
    ("Piso (mediana)", "HistGB (Huber Loss)"),
    ("Ridge", "Random Forest"),
    ("Ridge", "HistGB (Huber Loss)"),
    ("Random Forest", "HistGB (Huber Loss)")
]
print(pd.DataFrame([comparar(a, b) for a, b in pares]).round(4).to_string(index=False))

### ¿Qué pasaría si los pedazos se armaran al azar?

Misma comparación, pero con `KFold(5, shuffle=True)`: películas de todas las épocas mezcladas en entrenamiento y validación.

In [ ]:
# 9. TimeSeriesSplit vs. KFold al azar
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
folds_kf = evaluar(candidatos, kf)
contraste = pd.DataFrame({
    "MAE temporal": folds_tss.mean(),
    "MAE al azar": folds_kf.mean(),
    "desvío al azar": folds_kf.std(ddof=0),
})
contraste["diferencia"] = contraste["MAE temporal"] - contraste["MAE al azar"]
print(contraste.round(4).to_string())

## 6. Ajuste de hiperparámetros del modelo elegido

Se ajusta **Random Forest**: es el único modelo que le gana a Ridge en los 5 folds y por más que el desvío de la diferencia (sección 5). Grilla sobre los dos hiperparámetros que más controlan cuánto se ajusta a los datos:
- `min_samples_leaf`: cuántas películas como mínimo tiene cada hoja. Más alto, árboles más simples.
- `max_features`: qué fracción de las columnas mira cada corte. Más bajo, árboles más distintos entre sí.

Con los mismos 5 folds temporales y la tabla completa, no solo el ganador.

In [ ]:
# 10. Grilla con validación temporal
grilla = GridSearchCV(
    modelo(RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=SEED)),
    param_grid={"modelo__min_samples_leaf": [1, 5, 10, 20, 40],
                "modelo__max_features": [0.3, 0.6, 1.0]},
    cv=tss, scoring="neg_mean_absolute_error",
)
grilla.fit(X_train, y_train)

tabla_grilla = pd.DataFrame(grilla.cv_results_)[
    ["param_modelo__min_samples_leaf", "param_modelo__max_features", "mean_test_score", "std_test_score"]
].rename(columns={"param_modelo__min_samples_leaf": "min_samples_leaf",
                  "param_modelo__max_features": "max_features",
                  "mean_test_score": "MAE medio", "std_test_score": "desvío"})
tabla_grilla["MAE medio"] *= -1
tabla_grilla = tabla_grilla.sort_values("MAE medio").reset_index(drop=True)
print(tabla_grilla.round(4).to_string())
print(f"\nMejor combinación: {grilla.best_params_}")

# Comparación pareada del modelo ajustado contra el piso y contra Ridge, fold por fold
mejor = grilla.best_index_
folds_tss["Random Forest ajustado"] = [-grilla.cv_results_[f"split{i}_test_score"][mejor] for i in range(5)]
print("\n" + pd.DataFrame([comparar("Piso (mediana)", "Random Forest ajustado"),
                           comparar("Ridge", "Random Forest ajustado")]).round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
for mf, grupo in tabla_grilla.groupby("max_features"):
    grupo = grupo.sort_values("min_samples_leaf")
    ax.errorbar(grupo["min_samples_leaf"], grupo["MAE medio"], yerr=grupo["desvío"] / np.sqrt(5),
                marker="o", capsize=3, label=f"max_features = {mf}")
ax.axhline(resumen.loc["Piso (mediana)", "MAE medio"], color="crimson", linestyle="--", label="Piso")
ax.set_xscale("log")
ax.set_xlabel("min_samples_leaf (escala log)")
ax.set_ylabel("MAE medio en validación")
ax.set_title("Curva de validación de Random Forest (barras: error estándar)")
ax.legend()
plt.tight_layout()
plt.show()

**Clases desparejas:** no aplica. Es una regresión, no hay clases. Lo equivalente es la cola larga del objetivo, y eso se resolvió con la escala logarítmica (sección 2).

## 7. Columnas que probamos: con y sin cada grupo

Se saca un grupo de columnas por vez y se vuelve a validar el modelo ajustado con los mismos folds. Si al sacar un grupo el MAE **baja** o casi no cambia, ese grupo no aporta.

In [ ]:
# 11. Prueba de columnas: MAE sin cada grupo
mejor_rf = dict(n_estimators=300, n_jobs=-1, random_state=SEED,
                min_samples_leaf=grilla.best_params_["modelo__min_samples_leaf"],
                max_features=grilla.best_params_["modelo__max_features"])
grupos = {
    "(todas las columnas)": [],
    "sin budget": ["budget"],
    "sin popularidades (director y elenco)": ["director_popularity", "lead_actor_popularity", "co_star_popularity"],
    "sin productoras": COLS_PROD,
    "sin calendario (mes y día)": ["release_month", "release_day_of_week"],
    "sin idioma": ["original_language"],
    "sin géneros": ["primary_genre", "secondary_genre"],
    "sin runtime": ["runtime"],
    "sin release_year": ["release_year"],
}
filas = []
for nombre, excluir in grupos.items():
    r = cross_validate(modelo(RandomForestRegressor(**mejor_rf), excluir=excluir), X_train, y_train,
                       cv=tss, scoring="neg_mean_absolute_error")
    filas.append({"variante": nombre, "MAE medio": -r["test_score"].mean(), "desvío": r["test_score"].std()})
ablacion = pd.DataFrame(filas)
ablacion["cambio vs todas"] = ablacion["MAE medio"] - ablacion.loc[0, "MAE medio"]
print(ablacion.round(4).to_string(index=False))
print("\n(cambio positivo = sin ese grupo el modelo empeora, o sea que el grupo aporta)")

## 8. Modelo final, leído

Se entrena con todo el 80% de entrenamiento y se mide **una sola vez** sobre el test. Nada de lo que sigue se usa para cambiar el modelo.

In [ ]:
# 12. Medición única sobre el test
# Entrenamos el estimador con penalización de extremos y calculamos métricas integrales
modelo_final = candidatos["HistGB (Huber Loss)"].fit(X_train, y_train)
piso = DummyRegressor(strategy="median").fit(X_train, y_train)

pred_test = modelo_final.predict(X_test)
piso_pred_test = piso.predict(X_test)

mae_final = mean_absolute_error(y_test, pred_test)
mae_piso = mean_absolute_error(y_test, piso_pred_test)

rmse_final = root_mean_squared_error(y_test, pred_test)
rmse_piso = root_mean_squared_error(y_test, piso_pred_test)

r2_final = r2_score(y_test, pred_test)
r2_piso = r2_score(y_test, piso_pred_test)

std_real = y_test.std()
std_pred = pred_test.std()
cobertura_varianza = (std_pred / std_real) * 100

print(f"Test ({test['release_year'].min()}-{test['release_year'].max()}, {len(test):,} películas)")
print(f"  Piso:             MAE = {mae_piso:.4f}  | RMSE = {rmse_piso:.4f} | R² = {r2_piso:.4f} (factor {10 ** mae_piso:.2f})")
print(f"  HistGB (Huber):   MAE = {mae_final:.4f} | RMSE = {rmse_final:.4f} | R² = {r2_final:.4f} (factor {10 ** mae_final:.2f})")
print(f"  Mejora MAE:       {mae_piso - mae_final:.4f} ({(mae_piso - mae_final) / mae_piso:.1%} menos error)")
print(f"  Cobertura Varianza (Des-achatamiento): {cobertura_varianza:.1f}% de la dispersión real del target")

In [ ]:
# 13. Dónde se equivoca: real vs. predicho y cobertura de la distribución
res = test[["title", "release_year", "budget", "revenue", "roi", "vote_average", "vote_count"]].copy()
res["y_real"], res["y_pred"] = y_test.values, pred_test
res["roi_pred"] = 10 ** res["y_pred"] - 1
res["error_abs"] = (res["y_real"] - res["y_pred"]).abs()
res["tramo"] = pd.cut(res["roi"], [-1.01, 0, 1, 3, 10, np.inf],
                      labels=["Pérdida (ROI<0)", "Recupera hasta 2x", "2x a 4x", "4x a 11x", "Más de 11x"])
por_tramo = res.groupby("tramo", observed=True).agg(
    películas=("y_real", "size"), MAE=("error_abs", "mean"),
    ROI_real_mediano=("roi", "median"), ROI_predicho_mediano=("roi_pred", "median"),
    sesgo=("y_pred", lambda s: (s - res.loc[s.index, "y_real"]).mean()))
print(por_tramo.round(3).to_string())
print("\n(sesgo > 0: el modelo predice de más; sesgo < 0: predice de menos)")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: Scatter Real vs Predicho con línea de identidad y métricas
axes[0].scatter(res["y_real"], res["y_pred"], alpha=0.3, s=12, color="teal")
lim = [res["y_real"].min(), res["y_real"].max()]
axes[0].plot(lim, lim, color="crimson", linestyle="--", label="Predicción perfecta")
axes[0].axhline(piso.predict(X_test[:1])[0], color="gray", linestyle=":", label="Piso (mediana)")
axes[0].set_xlabel("Real: log10(revenue / budget)")
axes[0].set_ylabel("Predicho")
axes[0].set_title(f"Real vs. Predicho en test (R² = {r2_final:.3f} | MAE = {mae_final:.3f})")
axes[0].legend()

# Panel 2: Comparación de densidades para evidenciar que el modelo NO colapsa al centro
axes[1].hist(res["y_real"], bins=40, density=True, alpha=0.45, color="steelblue", label="Distribución Real (Test)")
axes[1].hist(res["y_pred"], bins=40, density=True, alpha=0.45, color="darkorange", label="Predicciones Modelo (Huber)")
axes[1].axvline(piso.predict(X_test[:1])[0], color="gray", linestyle=":", label="Colapso en Mediana (Piso)")
axes[1].set_title("Cobertura de Varianza (Des-achatamiento de la Curva)")
axes[1].set_xlabel("log10(revenue / budget)")
axes[1].set_ylabel("Densidad")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 14. Casos concretos: los errores más grandes del test
cols = ["title", "release_year", "budget", "revenue", "roi", "roi_pred"]
print("=== Predijo mucho más de lo que pasó (fracasos no anticipados) ===")
print(res.assign(e=res["y_pred"] - res["y_real"]).nlargest(5, "e")[cols].round(2).to_string(index=False))
print("\n=== Predijo mucho menos de lo que pasó (éxitos no anticipados) ===")
print(res.assign(e=res["y_real"] - res["y_pred"]).nlargest(5, "e")[cols].round(2).to_string(index=False))

### En qué se apoya: importancia por permutación

Se desordena una columna por vez en el test y se mide cuánto sube el MAE. Si sube mucho, el modelo depende de esa columna.

In [ ]:
# 15. Importancia por permutación sobre el test (evaluando las nuevas features de negocio)
imp = permutation_importance(modelo_final, X_test, y_test, scoring="neg_mean_absolute_error",
                             n_repeats=20, random_state=SEED, n_jobs=-1)
importancia = pd.DataFrame({"aumento del MAE": imp.importances_mean, "desvío": imp.importances_std},
                           index=FEATURES).sort_values("aumento del MAE", ascending=False)
print("=== Top 10 variables que más aportan al modelo ===")
print(importancia.head(10).round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
orden = importancia.head(12).iloc[::-1]
ax.barh(orden.index, orden["aumento del MAE"], xerr=orden["desvío"], color="teal", alpha=0.85)
ax.set_title("Importancia por permutación en test (Top 12 variables)")
ax.set_xlabel("Cuánto empeora el MAE al desordenar la columna")
plt.tight_layout()
plt.show()

### ¿La columna que más pesa es una fuga?

`budget` es la que más pesa, y conviene sospechar porque también está en el denominador del objetivo. No es una fuga: el presupuesto se conoce **antes** de filmar, y el modelo lo va a tener siempre que se use. Lo que aprende es real: las películas caras rara vez multiplican mucho su costo, y las de presupuesto bajo tienen más variación, con fracasos y también los grandes fenómenos (*Paranormal Activity*, *The Blair Witch Project*).

La sospecha de verdad está en las **popularidades**: TMDb devuelve la popularidad de la persona *hoy*, no la que tenía antes del rodaje. Si una película fue un éxito, su director y su elenco quedan más populares. Dos controles:

In [ ]:
# 16. Control de fuga en las popularidades
# a) ¿Las popularidades se asocian con el éxito ex post (vote_count)? Spearman sobre entrenamiento.
cols_pop = ["director_popularity", "lead_actor_popularity", "co_star_popularity"]
print("Spearman con vote_count (éxito después del estreno):")
print(train[cols_pop + ["vote_count"]].corr(method="spearman")["vote_count"][cols_pop].round(3).to_string())

# b) ¿Cuánto cambia el test si se sacan? Mismo modelo, sin las tres columnas.
sin_pop = modelo(RandomForestRegressor(**mejor_rf), excluir=cols_pop).fit(X_train, y_train)
mae_sin_pop = mean_absolute_error(y_test, sin_pop.predict(X_test))
print(f"\nMAE test con popularidades: {mae_final:.4f} | sin popularidades: {mae_sin_pop:.4f} | "
      f"diferencia: {mae_sin_pop - mae_final:+.4f}")

### Auditoría con la crítica del público (columnas post-estreno)

`vote_average` no entró al modelo. Ahora se usa para auditar: si las películas a las que el modelo les predijo mejor ROI también fueron mejor recibidas, el modelo capta algo de "calidad comercial" y no solo tamaño.

In [ ]:
# 17. Validación post-predicción con vote_average
res["cuartil_pred"] = pd.qcut(res["y_pred"], 4, labels=["Q1 (peor ROI predicho)", "Q2", "Q3", "Q4 (mejor ROI predicho)"])
print(res.groupby("cuartil_pred", observed=True).agg(
    películas=("roi", "size"), ROI_real_mediano=("roi", "median"),
    vote_average_medio=("vote_average", "mean")).round(3).to_string())
print(f"\nSpearman(ROI predicho, vote_average) = {res['y_pred'].corr(res['vote_average'], method='spearman'):.3f}")
print(f"Spearman(ROI real, vote_average)     = {res['y_real'].corr(res['vote_average'], method='spearman'):.3f}")

## 9. Transparencia: cuántas veces se miró el test

El test se miró **dos veces**. En la primera corrida, los errores más grandes eran películas con recaudaciones imposibles (*Memoir of a Snail* con 7 USD, *Society of the Snow* con 1.281 USD). Por eso se agregó el filtro `revenue >= 10.000 USD` y se volvió a correr.

Por qué lo consideramos aceptable, y por qué igual lo decimos:
- No cambió el modelo, los hiperparámetros ni las columnas: es una limpieza de montos imposibles, con el mismo umbral que ya se usaba para `budget`.
- El problema también estaba en el entrenamiento: hay recaudaciones menores a 10.000 USD en películas de 1971, 1985, 1999, 2006, 2009 y otros años. Se podía detectar sin mirar el test.
- Saca 22 películas de 6.522.
- Aun así, el número de test quedó un poco menos "virgen" de lo que debería. Para la Entrega 4 la limpieza queda fija y el test no se vuelve a tocar.